### Capstone Project-5: Multi-Agent Research Analyst - Building an Agentic AI System with LangGraph

#### 1.1 Import all the required libraries, packages

In [1]:
import os
import json
import time
import re
import ast
import operator
from typing import TypedDict, Annotated, List, Dict, Any, Callable
from dotenv import load_dotenv

from langchain_core.tools import tool
from langchain_groq import ChatGroq
from langchain_core.messages import HumanMessage
from langgraph.graph import StateGraph, START, END
from langgraph.checkpoint.memory import InMemorySaver
from langgraph.errors import GraphInterrupt
from langchain_tavily import TavilySearch, TavilyExtract

d:\G_AI\logicmojo-data-science-ai-nov-2025\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
load_dotenv() # This line loads the variables and values from the .env file
llm = ChatGroq(model="llama-3.3-70b-versatile", temperature=0)

#### 1.2 Define Tools
##### Tools: Tools is a function or method which takes defined inputs/arguments and returns a result or performs an action.. Tools Acts as a hands which perform some actions(get data, extract data, hitting API requests) in external apps or systems.
- web_search() : This is the tool which is used to get the latest Details from the internet.
Why use it ? Because LLMS are trained till a specific date eg: GPT 5 Knowledge/Data cutoff is Dec 31st 2025
- Calculation() : This is used to do math calculations LLMS are bad at math calculation because they are next word predictors mainly.
- Extract_context() : Extract the full text content from a web page.
#### Wikipedia Tool: INTENTIONALLY REMOVED.
- The underlying wikipedia Python library has a bug
- Tavily web_search is a required tool and searches the whole internet (including Wikipedia). It makes the Wikipedia tool redundant.

In [3]:
"""
Tool Layer for the Multi-Agent Research Analyst

Each tool is a function decorated with @tool.
The docstring is NOT documentation for humans - it's PROMPT for the LLM.
Write it as if you're explaining to a smart assistant who has never seen this tool.
"""


TAVILY_KEY = os.getenv("TAVILY_KEY")


# ============================================================================
# TOOL 1: WEB SEARCH 
# ============================================================================

from langchain_tavily import TavilySearch
import os

@tool
def web_search(query: str) -> str: 
    """
    Search the web for current information about a topic.
    
    Use this tool when you need to:
    - Find recent news, articles, or reports
    - Get up-to-date information (post-training cutoff)
    - Discover multiple perspectives on a topic
    - Find specific data points or statistics
    
    Args:
        query: A focused search query. Be specific — "Tata Motors EV sales 2024" 
               is better than "Tata Motors"
    
    Returns:
        A formatted text string containing numbered search results with titles, URLs, and snippets.
    """
    # 1. Force pass the API key
    api_key = os.getenv("TAVILY_KEY") # Keeping the exact variable name
    
    # 2. Initialize (Hardcoded to 5 so the LLM never sees it)
    search_tool = TavilySearch(max_results=5, tavily_api_key=api_key)
    
    # 3. Invoke the search
    response_dict = search_tool.invoke(query)
    
    # 4. Extract the actual list from inside the dictionary
    if isinstance(response_dict, dict):
        raw_results = response_dict.get("results", [])
    else:
        raw_results = response_dict 
        
    # 5. Format as a clean STRING (LLMs read text better than Python dicts)
    if not raw_results:
        return f"No results found for '{query}'"
        
    output_lines = []
    for i, item in enumerate(raw_results, 1):
        title = item.get("title", "No title")
        url = item.get("url", "")
        content = item.get("content", "No content available")[:200] # Truncated to save tokens
        output_lines.append(f"{i}. {title}\n   URL: {url}\n   Snippet: {content}")
        
    return "\n\n".join(output_lines)


@tool
def extract_page(url: str) -> str:
    """
    Extract the full text content from a web page.
    
    Use this tool when:
    - A search result snippet seems promising but lacks detail
    - You need the full context of an article, not just a summary
    
    Args:
        url: The full URL of the page to extract
    
    Returns:
        The extracted text content of the page
    """
    try:
        api_key = os.getenv("TAVILY_KEY")
        extractor = TavilyExtract(depth="advanced", TAVILY_KEY=api_key)
        
        # TavilyExtract expects a list of URLs in a dict
        results = extractor.invoke({"urls": [url]})
        
        # It returns a list of dicts
        if results and len(results) > 0:
            content = results[0].get("raw_content", "")
            if content:
                if len(content) > 15000:
                    content = content[:15000] + "\n\n[CONTENT TRUNCATED]"
                return content
            return "Error: No content extracted from this page."
        return "Error: Extraction returned no results."
    
    except Exception as e:
        return f"Error extracting page: {str(e)}"


# ============================================================================
# TOOL 3: SAFE CALCULATOR
# ============================================================================

@tool
def calculator(expression: str) -> str:
    """
    Perform safe arithmetic calculations.
    
    Use this tool when you need to:
    - Calculate percentages, growth rates, or ratios
    - Add, subtract, multiply, or divide numbers found in research
    - Convert between units
    - Compare numerical values
    
    This tool ONLY supports basic arithmetic: +, -, *, /, **, (), and math functions.
    It does NOT support variable assignment, imports, or any code execution.
    
    Args:
        expression: A mathematical expression as a string.
                   Examples: "15000 * 0.12", "(4500 - 3800) / 3800 * 100"
    
    Returns:
        The calculated result as a string, or an error message
    
    Example:
        calculator("(4500 - 3800) / 3800 * 100")  # Growth rate calculation
        → "18.421052631578945"
    """
    # Whitelist of safe operations
    allowed_names = {
        'abs': abs,
        'round': round,
        'min': min,
        'max': max,
        'sum': sum,
        'pow': pow,
        'len': len,
    }
    
    try:
        # Parse the expression into an AST
        tree = ast.parse(expression, mode='eval')
        
        # Walk the AST and check for disallowed operations
        for node in ast.walk(tree):
            # Disallow function calls (except our whitelisted ones)
            if isinstance(node, ast.Call):
                if not isinstance(node.func, ast.Name) or node.func.id not in allowed_names:
                    raise ValueError(f"Function call not allowed: {ast.dump(node)}")
            
            # Disallow attribute access (no obj.method())
            if isinstance(node, ast.Attribute):
                raise ValueError(f"Attribute access not allowed: {ast.dump(node)}")
            
            # Disallow imports
            if isinstance(node, (ast.Import, ast.ImportFrom)):
                raise ValueError("Imports not allowed")
        
        # Compile and evaluate safely
        code = compile(tree, '<string>', 'eval')
        result = eval(code, {"__builtins__": {}}, allowed_names)
        
        return str(result)
    
    except SyntaxError as e:
        return f"Syntax error in expression: {e}"
    except ValueError as e:
        return f"Security error: {e}"
    except Exception as e:
        return f"Calculation error: {e}"


# ============================================================================
# TOOL 4: WIKIPEDIA
# ============================================================================

@tool  
def wikipedia_search(query: str) -> str:
    """
    Search Wikipedia for background information and definitions.
    
    Use this tool when:
    - You need basic definitions or historical context
    - You want to understand a concept before searching for recent info
    - The topic is well-established and likely has a good Wikipedia article
    
    DO NOT use this tool for:
    - Recent news or current events (use web_search instead)
    - Topics that are too niche for Wikipedia
    
    Args:
        query: Search term to look up on Wikipedia
    
    Returns:
        A summary of the Wikipedia article, or an error message
    
    Example:
        wikipedia_search("BYD Company")
        → "BYD Company is a Chinese manufacturing company..."
    """
    try:
        from wikipedia import summary as wiki_summary
        result = wiki_summary(query, sentences=8)
        return result
    except Exception as e:
        return f"Wikipedia search failed: {str(e)}. Try web_search for this topic."


# ============================================================================
# EXPORT ALL TOOLS
# ============================================================================

# This is the list of tools we'll give to researcher agents
RESEARCHER_TOOLS = [web_search, extract_page, calculator]

# Full tool set (including optional ones)
ALL_TOOLS = [web_search, extract_page, calculator, wikipedia_search]



#### Tool Testing 

In [35]:
print("Testing Calculator : ")
print(calculator.invoke({"expression" : "(500 - 400) / 400 * 100"}), "% growth")

Testing Calculator : 
25.0 % growth


In [36]:
print("Testing Web Search: ")
print(web_search.invoke({"query" : "Check Details about Bruce Lee ?"}))

Testing Web Search: 
1. Bruce Lee: Biography, Actor, Martial Arts Expert
   URL: https://www.biography.com/actors/bruce-lee
   Snippet: Bruce Lee was a groundbreaking actor, director, and martial arts expert known for his roles in movies such as The Chinese Connection and Enter the Dragon. Born in the United States before becoming a c

2. 11 Amazing Facts About Bruce Lee — Google Arts & Culture
   URL: https://artsandculture.google.com/story/11-amazing-facts-about-bruce-lee/QwVR7wlTt5sELw?hl=en
   Snippet: There are many facts about Bruce Lee that are well-known: that he was the first Asian-American actor to ever have a lead role in a Hollywood film; that he was nicknamed ‘Little Dragon’ (he was born on

3. Bruce Lee
   URL: https://brucelee.com/bruce-lee
   Snippet: ## Short Bio

Bruce Jun Fan Lee (Lee Siu Loong) was born in 1940 in San Francisco, CA while his parents were on tour with the Chinese Opera. Ultimately raised in Hong Kong, Bruce Lee was a child actor

4. Bruce Lee | Biogr

In [49]:
print("Testing wikipedia:")
print(wikipedia_search.invoke({"query" : "Explain about bruce lee philosophy and books ?"}))

Testing wikipedia:
Wikipedia search failed: Expecting value: line 1 column 1 (char 0). Try web_search for this topic.


#### State : Data or Input we pass to agent graph each step in the agent graph will update each item/attribute/variable of the state.
#### why state ?
- Because Agents nodes take query as input and give the reasoned or logical answer as output. As the functions varibales scope is local to function and will be vanished in RAM So we need to save the state in some place that is the reason state is used. 

#### GPT Answer : 
- State is the shared data/context of an agent graph. Each node reads the information it needs from the state and returns updates to the state. This allows different nodes to communicate and progressively build the result as the graph executes.

In [4]:
"""
STEP 4: SHARED STATE DEFINITION

This is the MOST IMPORTANT file in multi-agent system.
Every agent reads from and writes to this state.

KEY CONCEPTS:
1. TypedDict: Defines the "shape" of our whiteboard (what columns exist).
2. Annotated[X, reducer]: Tells LangGraph HOW to handle multiple writes to column X.
"""

class ResearchState(TypedDict):
    """
    The shared whiteboard for our multi-agent system.
    """
    
    # ========================================================================
    # INPUT (Written once by the user, read by everyone)
    # ========================================================================
    question: str
    """The original research question from the user."""
    
    
    # ========================================================================
    # PLANNING PHASE (Written by Planner, read by everyone)
    # ========================================================================
    plan: List[str]
    """
    List of sub-questions (e.g., ["What is Tata's market share?", "What is BYD's strategy?"]).
    NO REDUCER NEEDED: Only the Planner writes to this, so there's no collision risk.
    """
    
    
    # ========================================================================
    # RESEARCH PHASE (Written by PARALLEL Researchers) - NEEDS REDUCER!
    # ========================================================================
    findings: Annotated[List[Dict[str, Any]], operator.add]
    """
    A list of research findings from the researchers.
    
    WHY OPERATOR.ADD?
    If Researcher 1 returns [{"text": "Tata..."}] and Researcher 2 returns [{"text": "BYD..."}],
    LangGraph will automatically combine them into:
    [{"text": "Tata..."}, {"text": "BYD..."}]
    
    Without this, whoever finishes LAST would overwrite everyone else!
    """
    
    
    # ========================================================================
    # SYNTHESIS PHASE (Written by Synthesiser, read by Critic)
    # ========================================================================
    draft: str
    """The combined research brief written by the Synthesiser. 
    NO REDUCER: Only one writer at this stage."""
    
    references: List[Dict[str, str]]
    """List of citations [{'index': 1, 'url': '...'}]. 
    NO REDUCER: Only one writer."""
    
    
    # ========================================================================
    # CRITIQUE PHASE (Written by Critic, used for routing)
    # ========================================================================
    critique: str
    """The Critic's feedback. Either 'APPROVED' or a list of gaps."""
    
    iterations: int
    """How many times we've gone through the Critic loop. Used to prevent infinite loops."""
    
    approved: bool
    """True if the Critic approves the draft. False otherwise."""


#### Nodes

- **Node:** A node is a unit of work in an agent graph. Programmatically, it is typically a function that receives the current graph state, performs an operation such as calling an LLM, tool, API, database, or Python logic, and returns updates to the state.

In [5]:
# Agents Code

"""
Specialist Sub-Agents for the Multi-Agent Research System.

This module defines the core logic for each node in the LangGraph workflow.
Each function takes the current ResearchState, performs a specific task,
and returns a partial dictionary to update the shared state.
"""

# Maximum allowed iterations for the Critic loop to prevent infinite runs
MAX_CRITIC_ITERATIONS = 2


# ============================================================================
# NODE 1: PLANNER
# ============================================================================

PLANNER_SYSTEM_PROMPT = """You are an expert research planner. Your task is to decompose a complex research question into 3 to 5 distinct, searchable sub-questions.

RULES:
1. Each sub-question must be specific enough to be answered by a single web search.
2. Sub-questions must not overlap in scope.
3. Together, the sub-questions must comprehensively cover the original question.
4. Respond ONLY with a valid JSON array of strings. No markdown, no explanation.

EXAMPLE INPUT: What is the investment case for green hydrogen in India?
EXAMPLE OUTPUT: ["What are the current government subsidies for green hydrogen in India?", "Who are the key private players investing in Indian green hydrogen?", "What are the main risks and cost challenges for green hydrogen production in India?"]
"""

def planner_node(state: ResearchState) -> dict:
    """
    Decomposes the main question into a list of sub-questions.
    
    Args:
        state: The current shared state containing the user's question.
        
    Returns:
        A dictionary updating the 'plan', 'iterations', and 'approved' keys.
    """
    print(f"[PLANNER] Decomposing question: {state['question']}")
    
    response = llm.invoke(
        [HumanMessage(content=f"{PLANNER_SYSTEM_PROMPT}\n\nQUESTION: {state['question']}")]
    )
    
    raw_content = response.content.strip()
    
    # Robust JSON extraction: LLMs sometimes wrap JSON in markdown code blocks
    json_match = re.search(r"\[.*\]", raw_content, re.DOTALL)
    if json_match:
        try:
            plan = json.loads(json_match.group(0))
            if not isinstance(plan, list):
                plan = [plan]
        except json.JSONDecodeError:
            plan = _generate_fallback_plan(state["question"])
    else:
        plan = _generate_fallback_plan(state["question"])
        
    # Ensure we don't exceed 5 sub-questions
    plan = plan[:5]
    print(f"[PLANNER] Generated {len(plan)} sub-questions.")
    
    return {
        "plan": plan,
        "iterations": 0,
        "approved": False
    }


def _generate_fallback_plan(question: str) -> List[str]:
    """Generates a generic fallback plan if JSON parsing fails."""
    return [
        f"What is the current state of {question}?",
        f"What are the key drivers or causes related to {question}?",
        f"What are the future projections or solutions for {question}?"
    ]


# ============================================================================
# NODE 2: RESEARCHER (Agent Node with Tools)
# ============================================================================

RESEARCHER_PROMPT_HEADER = """You are a focused research specialist. Answer the sub-question using the available tools.

Available Tools:
- web_search: Takes a {{"query": "string"}} to search the web.
- extract_page: Takes a {{"url": "string"}} to read a full webpage.
- calculator: Takes an {{"expression": "string"}} to do math.

You MUST use the following format strictly:
Question: {question}
Thought: I need to search for...
Action: web_search
Action Input: {{"query": "my search query"}}
Observation: [Tool output will appear here]
... (repeat as needed)
Thought: I now have the final answer.
Final Answer: [Write your detailed 2-3 paragraph answer here, citing sources like [1], [2]]"""


def _run_manual_react_loop(sub_question: str) -> str:
    """
    A custom, pure-Python implementation of the ReAct agent loop.
    Includes retry logic to handle Groq free-tier rate limits (Error 429).
    """
    tools_map = {
        "web_search": web_search,
        "extract_page": extract_page,
        "calculator": calculator
    }
    
    prompt = RESEARCHER_PROMPT_HEADER.format(question=sub_question)
    messages = [{"role": "user", "content": prompt}]
    
    for _ in range(6): # Max 6 tool loops
        # --- RETRY LOGIC FOR RATE LIMITS ---
        max_retries = 5
        for attempt in range(max_retries):
            try:
                response = llm.invoke(messages)
                break # Success! Exit the retry loop
            except Exception as e:
                error_str = str(e)
                if "429" in error_str or "rate_limit" in error_str:
                    wait_time = 10 * (attempt + 1) # Wait 10s, 20s, 30s, etc.
                    print(f"[RATE LIMIT] Groq TPM limit hit. Waiting {wait_time} seconds before retry...")
                    time.sleep(wait_time)
                else:
                    raise e # If it's not a rate limit, crash normally
        else:
            return "Error: Failed to get LLM response after multiple retries due to rate limits."
        # -------------------------------------

        text = response.content
        
        if "Final Answer:" in text:
            return text.split("Final Answer:")[-1].strip()
            
        action_match = re.search(r"Action: (\w+)", text)
        input_match = re.search(r"Action Input: ({.*?})", text, re.DOTALL)
        
        if not action_match or not input_match:
            return text
            
        tool_name = action_match.group(1)
        
        try:
            tool_input = json.loads(input_match.group(1))
        except json.JSONDecodeError:
            return text
            
        if tool_name not in tools_map:
            observation = f"Error: Tool '{tool_name}' does not exist."
        else:
            try:
                observation = tools_map[tool_name].invoke(tool_input)
            except Exception as e:
                observation = f"Error executing tool: {str(e)}"
                
        messages.append({"role": "assistant", "content": text})
        messages.append({"role": "user", "content": f"Observation: {observation}\nThought:"})
        
    return text



def create_researcher_node(researcher_id: int) -> Callable:
    """
    Factory function to create a researcher node.
    """
    def researcher_node(state: ResearchState) -> dict:
        plan = state.get("plan", [])
        
        if researcher_id >= len(plan):
            print(f"[RESEARCHER {researcher_id + 1}] No sub-question assigned. Skipping.")
            return {"findings": []}
            
        sub_question = plan[researcher_id]
        print(f"[RESEARCHER {researcher_id + 1}] Investigating: {sub_question}")
        
        # Run our custom loop instead of an agent factory
        answer = _run_manual_react_loop(sub_question)
        
        finding = {
            "researcher_id": researcher_id + 1,
            "sub_question": sub_question,
            "answer": answer
        }
        
        print(f"[RESEARCHER {researcher_id + 1}] Completed investigation.")
        return {"findings": [finding]}
        
    return researcher_node


# ============================================================================
# NODE 3: SYNTHESISER
# ============================================================================

SYNTHESISER_PROMPT = """You are an expert technical writer. Your job is to synthesize multiple research findings into a single, cohesive, professional research brief.

ORIGINAL QUESTION: {question}

RESEARCH FINDINGS:
{findings_text}

INSTRUCTIONS:
1. Do not just list the findings. Weave them into a logical narrative.
2. Organize the brief with clear headings (##).
3. In your text, use inline citations like [1], [2], etc., to refer to the sources provided in the findings.
4. End the brief with a "## References" section. Map the citations [1], [2], etc., to the actual URLs provided in the findings.
5. Be objective and analytical. If findings contradict, acknowledge it."""

def synthesiser_node(state: ResearchState) -> dict:
    """
    Combines all parallel findings into a single drafted brief.
    
    Args:
        state: The current shared state containing all gathered findings.
        
    Returns:
        A dictionary updating the 'draft' and 'references' keys.
    """
    print("[SYNTHESISER] Writing research brief...")
    
    findings_text = ""
    for i, finding in enumerate(state.get("findings", []), 1):
        findings_text += f"--- FINDING {i} ---\n"
        findings_text += f"Sub-Question: {finding['sub_question']}\n"
        findings_text += f"Answer: {finding['answer']}\n\n"
        
    response = llm.invoke(
        [HumanMessage(content=SYNTHESISER_PROMPT.format(
            question=state["question"],
            findings_text=findings_text
        ))]
    )
    
    draft = response.content
    
    # Build a basic reference list from the findings for metadata
    references = [
        {"id": i+1, "question": f["sub_question"]} 
        for i, f in enumerate(state.get("findings", []))
    ]
    
    print("[SYNTHESISER] Draft complete.")
    return {"draft": draft, "references": references}


# ============================================================================
# NODE 4: CRITIC
# ============================================================================

CRITIC_PROMPT = """You are a rigorous quality control editor. Evaluate the following research brief.

ORIGINAL QUESTION: {question}

RESEARCH BRIEF:
{draft}

EVALUATION CRITERIA:
1. Coverage: Does it answer all parts of the original question?
2. Faithfulness: Are all claims supported by the citations? (No hallucinations?)
3. Structure: Is it well-organized and readable?

OUTPUT INSTRUCTIONS:
If the brief is excellent and requires no changes, respond with exactly one word: APPROVED
If the brief has gaps, missing citations, or hallucinations, respond with: 
NEEDS_REVISION: [Provide a specific, actionable list of what is missing or wrong]"""

def critic_node(state: ResearchState) -> dict:
    """
    Evaluates the draft and decides if it is ready for delivery.
    
    Args:
        state: The current shared state containing the draft.
        
    Returns:
        A dictionary updating the 'critique', 'approved', and 'iterations' keys.
    """
    current_iterations = state.get("iterations", 0) + 1
    print(f"[CRITIC] Evaluating draft (Iteration {current_iterations}/{MAX_CRITIC_ITERATIONS})...")
    
    response = llm.invoke(
        [HumanMessage(content=CRITIC_PROMPT.format(
            question=state["question"],
            draft=state["draft"]
        ))]
    )
    
    critique = response.content.strip()
    is_approved = "APPROVED" in critique.upper() and "NEEDS_REVISION" not in critique.upper()
    
    if is_approved:
        print("[CRITIC] Decision: APPROVED")
    else:
        print(f"[CRITIC] Decision: NEEDS REVISION")
        print(f"[CRITIC] Feedback: {critique[:200]}...")
        
    return {
        "critique": critique,
        "approved": is_approved,
        "iterations": current_iterations
    }


# ============================================================================
# HELPER FUNCTIONS
# ============================================================================

def _extract_final_answer(messages: List[Any]) -> str:
    """
    Parses the LangGraph message history to find the final text response.
    """
    for msg in reversed(messages):
        if hasattr(msg, 'type') and msg.type == 'ai':
            if hasattr(msg, 'tool_calls') and msg.tool_calls:
                continue # Skip intermediate tool-calling messages
            if hasattr(msg, 'content') and msg.content.strip():
                return msg.content
    return "Error: Agent failed to produce a final text answer."


def should_continue(state: ResearchState) -> str:
    """
    Conditional edge routing function.
    Determines if the workflow should loop back to the synthesiser or end.
    
    Args:
        state: The current shared state.
        
    Returns:
        "revise" if the draft needs work and we haven't hit the max iterations.
        "end" if approved or max iterations reached.
    """
    if state.get("approved", False):
        return "end"
        
    if state.get("iterations", 0) >= MAX_CRITIC_ITERATIONS:
        print("[ROUTER] Max iterations reached. Proceeding with current draft.")
        return "end"
        
    return "revise"

In [6]:
# Graph Code

"""
LangGraph Orchestration Layer.

This module defines the state graph structure. It connects the specialist 
agents (nodes) using edges and conditional routing to execute the research pipeline.
"""

def build_research_graph(checkpointer=None, enable_interrupt=True) -> StateGraph:
    """
    Constructs and compiles the multi-agent research workflow.
    
    Architecture Patterns Used:
    - Planner-Executor: Planner creates the plan, graph executes it.
    - Orchestrator-Workers: Graph fans out to parallel researchers.
    - Evaluator-Optimizer: Critic loops the draft back to synthesiser.
    
    Args:
        checkpointer: Optional InMemorySaver instance for state persistence.
        enable_interrupt: If True, pauses before researchers for Human-in-the-Loop.
                         Set to False for automated web deployments (Streamlit).
    
    Returns:
        A compiled LangGraph StateGraph ready for invocation.
    """
    
    # Initialize the graph with our strict state schema
    workflow = StateGraph(ResearchState)
    
    # ========================================================================
    # 1. ADD NODES
    # ========================================================================
    
    workflow.add_node("planner", planner_node)
    workflow.add_node("synthesiser", synthesiser_node)
    workflow.add_node("critic", critic_node)
    
    workflow.add_node("researcher_1", create_researcher_node(0))
    workflow.add_node("researcher_2", create_researcher_node(1))
    workflow.add_node("researcher_3", create_researcher_node(2))
    
    # ========================================================================
    # 2. ADD EDGES (The Wiring)
    # ========================================================================
    
    workflow.add_edge(START, "planner")
    
    workflow.add_edge("planner", "researcher_1")
    workflow.add_edge("planner", "researcher_2")
    workflow.add_edge("planner", "researcher_3")
    
    workflow.add_edge("researcher_1", "synthesiser")
    workflow.add_edge("researcher_2", "synthesiser")
    workflow.add_edge("researcher_3", "synthesiser")
    
    workflow.add_edge("synthesiser", "critic")
    
    workflow.add_conditional_edges(
        "critic",
        should_continue,
        {
            "revise": "synthesiser",
            "end": END
        }
    )
    
    # ========================================================================
    # 3. COMPILE WITH PERSISTENCE AND CONDITIONAL INTERRUPT
    # ========================================================================
    
    if checkpointer is None:
        checkpointer = InMemorySaver()
        
    compile_args = {"checkpointer": checkpointer}
    
    if enable_interrupt:
        compile_args["interrupt_before"] = ["researcher_1"]
        
    return workflow.compile(**compile_args)

In [7]:
# Graph Visualize

# Visualize the graph structure
graph = build_research_graph()
print(graph.get_graph().draw_mermaid())

---
config:
  flowchart:
    curve: linear
---
graph TD;
	__start__([<p>__start__</p>]):::first
	planner(planner)
	synthesiser(synthesiser)
	critic(critic)
	researcher_1(researcher_1<hr/><small><em>__interrupt = before</em></small>)
	researcher_2(researcher_2)
	researcher_3(researcher_3)
	__end__([<p>__end__</p>]):::last
	__start__ --> planner;
	critic -. &nbsp;end&nbsp; .-> __end__;
	critic -. &nbsp;revise&nbsp; .-> synthesiser;
	planner --> researcher_1;
	planner --> researcher_2;
	planner --> researcher_3;
	researcher_1 --> synthesiser;
	researcher_2 --> synthesiser;
	researcher_3 --> synthesiser;
	synthesiser --> critic;
	classDef default fill:#f2f0ff,line-height:1.2
	classDef first fill-opacity:0
	classDef last fill:#bfb6fc



In [11]:
# Attempt to save as PNG
try:
    png_data = graph.get_graph().draw_mermaid_png()
    with open("graph_structure.png", "wb") as f:
        f.write(png_data)
    print("Graph visualization saved to graph_structure.png")
except Exception:
    print("Note: Could not generate PNG. Install graphviz system package to enable image generation.")

Graph visualization saved to graph_structure.png


In [8]:
# BaseLine Code

# THE ANTI-XML PROMPT
BASELINE_SYSTEM_PROMPT = """You are an expert research assistant. Answer the user's question thoroughly and accurately.

RULES:
1. You MUST use the 'web_search' tool to find current information before answering. Do not rely on your training data alone.
2. If you need to calculate a percentage, growth rate, or sum, use the 'calculator' tool.
3. In your final answer, cite your sources using [1], [2], etc., matching the search results you found.
4. Do not guess or make up URLs. Only use the information returned by the tools.
5. Provide a detailed, well-structured answer.

CRITICAL FORMATTING RULE: When you decide to use a tool, you MUST use the standard JSON function-calling format provided by the system. NEVER output XML tags like <function=...>. If you output XML tags, the system will crash and you will fail the task."""

def create_baseline_agent():
    return create_agent(
        model=llm,
        tools=[web_search, calculator],
        system_prompt=BASELINE_SYSTEM_PROMPT
    )

def run_baseline(question: str) -> dict:
    print("\n" + "=" * 70)
    print("BASELINE AGENT - Single Agent with Web Search")
    print("=" * 70)
    print(f"Question: {question}")
    print("-" * 70)
    
    agent = create_baseline_agent()
    
    start_time = time.time()
    result = agent.invoke({"messages": [HumanMessage(content=question)]})
    end_time = time.time()
    
    messages = result.get("messages", [])
    final_answer = "Agent did not produce a final answer."
    tool_steps = 0
    
    for msg in reversed(messages):
        if hasattr(msg, 'tool_calls') and msg.tool_calls:
            tool_steps += 1
        elif hasattr(msg, 'content') and msg.type == 'ai' and msg.content.strip():
            final_answer = msg.content
            break
            
    return {
        "question": question,
        "answer": final_answer,
        "time_seconds": round(end_time - start_time, 2),
        "steps": tool_steps
    }

In [47]:
test_questions = [
        "Compare the current EV strategies of Tata Motors and BYD.",
        "What are the main causes and latest policy responses to urban air pollution in Delhi?",
        "Summarize the 2024-2025 advances in small open-weight language models."
    ]
    
baseline_results = []
    
for q in test_questions:
    result = run_baseline(q)
    baseline_results.append(result)
    
    print("\n" + "-" * 70)
    print("ANSWER PREVIEW:")
    answer_text = result["answer"]
    print(answer_text[:1500] + "..." if len(answer_text) > 1500 else answer_text)
    print(f"\n Time: {result['time_seconds']}s | Tool Calls: {result['steps']}")

with open("baseline_results.json", "w") as f:
    json.dump(baseline_results, f, indent=2)

print("\n" + "=" * 70)
print("BASELINE COMPLETE. Results saved to baseline_results.json")
print("=" * 70)


BASELINE AGENT - Single Agent with Web Search
Question: Compare the current EV strategies of Tata Motors and BYD.
----------------------------------------------------------------------

----------------------------------------------------------------------
ANSWER PREVIEW:
To compare the current EV strategies of Tata Motors and BYD, I will need to search for the latest information on the web.

{"type": "function", "name": "web_search", "parameters": {"query": "Tata Motors EV strategy 2024"}} 
{"type": "function", "name": "web_search", "parameters": {"query": "BYD EV strategy 2024"}} 

Please wait for a moment while I search for the latest information. 

Once I have the latest information, I will be able to provide a detailed comparison of the current EV strategies of Tata Motors and BYD. 

After searching, I will provide a detailed answer with proper citations. 

Please give me a moment to search and compile the information. 

Based on the search results, I will provide a detailed comp

In [9]:
def demonstrate_human_loop():
    question = "Assess the investment case for green hydrogen in India."
    
    # 1. Initialize checkpointer and build graph
    memory = InMemorySaver()
    graph = build_research_graph(checkpointer=memory)
    
    thread_id = "human-demo-session-001"
    config = {"configurable": {"thread_id": thread_id}}
    
    initial_state = {
        "question": question, "plan": [], "findings": [], "draft": "",
        "references": [], "critique": "", "iterations": 0, "approved": False
    }
    
    # ========================================================================
    # STAGE 1: RUN PLANNER AND PAUSE
    # ========================================================================
    print("=" * 70)
    print("STAGE 1: RUNNING PLANNER")
    print("=" * 70)
    
    try:
        graph.invoke(initial_state, config)
    except GraphInterrupt:
        pass # Silently catch the pause
    
    # ========================================================================
    # STAGE 2: INSPECT THE SAVED STATE (Cleanly)
    # ========================================================================
    print("\n" + "=" * 70)
    print("STAGE 2: INSPECTING SAVED STATE")
    print("=" * 70)
    
    snapshot = memory.get(config)
    current_plan = snapshot["channel_values"]["plan"]
    
    print(f"\nPlanner generated {len(current_plan)} sub-questions:")
    for i, q in enumerate(current_plan, 1):
        print(f"  {i}. {q}")
        
    # ========================================================================
    # STAGE 3: REAL HUMAN INTERACTION
    # ========================================================================
    print("\n" + "=" * 70)
    print("STAGE 3: HUMAN INTERVENTION")
    print("=" * 70)
    
    action = input("\nType 'yes' to approve, or 'edit' to add a question: ").strip().lower()
    
    if action == 'edit':
        new_question = input("Type the new sub-question to add: ").strip()
        edited_plan = current_plan.copy()
        edited_plan.append(new_question)
        
        # THE CORRECT WAY: Update the paused state without restarting the graph
        graph.update_state(config, {"plan": edited_plan})
        print(f"\n[SYSTEM] Plan updated to {len(edited_plan)} questions. Resuming...")
    else:
        print("\n[SYSTEM] Plan approved. Resuming...")
        
    # ========================================================================
    # STAGE 4: RESUME GRAPH
    # ========================================================================
    print("\n" + "=" * 70)
    print("STAGE 4: RESUMING GRAPH")
    # ========================================================================
    
    # Pass None to tell LangGraph "just resume where you left off"
    final_state = graph.invoke(None, config)
    
    print("\n" + "=" * 70)
    print("SYSTEM COMPLETED")
    print("=" * 70)
    print(f"Final draft length: {len(final_state.get('draft', ''))} characters")
    print(f"Critic iterations: {final_state.get('iterations', 0)}")

In [43]:
# Run the human-in-the-loop demonstration
# When it pauses, type 'yes' in the output box below to approve the plan
demonstrate_human_loop()

STAGE 1: RUNNING PLANNER
[PLANNER] Decomposing question: Assess the investment case for green hydrogen in India.
[PLANNER] Generated 4 sub-questions.

STAGE 2: INSPECTING SAVED STATE

Planner generated 4 sub-questions:
  1. What are the current government policies and incentives for green hydrogen production in India?
  2. What is the estimated cost of green hydrogen production in India compared to traditional hydrogen production methods?
  3. Who are the major companies and investors currently involved in India's green hydrogen market?
  4. What are the primary applications and demand drivers for green hydrogen in India?

STAGE 3: HUMAN INTERVENTION

[SYSTEM] Plan approved. Resuming...

STAGE 4: RESUMING GRAPH
[RESEARCHER 1] Investigating: What are the current government policies and incentives for green hydrogen production in India?
[RESEARCHER 2] Investigating: What is the estimated cost of green hydrogen production in India compared to traditional hydrogen production methods?
[RESE

In [44]:
# Run the multi-agent system on one question
graph = build_research_graph(enable_interrupt=True)
state = {
    "question": "What is the current state of India's semiconductor manufacturing push?",
    "plan": [], "findings": [], "draft": "", "references": [],
    "critique": "", "iterations": 0, "approved": False
}

start = time.time()
final_state = graph.invoke(state, config={"configurable": {"thread_id": "notebook-test"}})
print(f"Time taken: {round(time.time() - start, 2)}s")
print(f"Critic Iterations: {final_state['iterations']}")
print("\n--- MULTI-AGENT BRIEF ---")
print(final_state["draft"])

[PLANNER] Decomposing question: What is the current state of India's semiconductor manufacturing push?
[PLANNER] Generated 3 sub-questions.
[RESEARCHER 1] Investigating: What are the key government initiatives and policies supporting semiconductor manufacturing in India?
[RESEARCHER 2] Investigating: Which international companies have established or plan to establish semiconductor manufacturing facilities in India?
[RESEARCHER 3] Investigating: What are the current challenges and infrastructure gaps hindering the growth of India's semiconductor industry?
[RESEARCHER 3] Completed investigation.
[RESEARCHER 1] Completed investigation.
[RESEARCHER 2] Completed investigation.
[SYNTHESISER] Writing research brief...
[SYNTHESISER] Draft complete.
[CRITIC] Evaluating draft (Iteration 1/2)...
[CRITIC] Decision: NEEDS REVISION
[CRITIC] Feedback: NEEDS_REVISION: 
1. The brief lacks specific details on the current state of India's semiconductor manufacturing capabilities, such as the number of ex

In [9]:
def build_research_graph(checkpointer=None, enable_interrupt=True) -> StateGraph:
    """
    Constructs and compiles the multi-agent research workflow.
    
    Architecture Patterns Used:
    - Planner-Executor: Planner creates the plan, graph executes it.
    - Orchestrator-Workers: Graph fans out to parallel researchers.
    - Evaluator-Optimizer: Critic loops the draft back to synthesiser.
    
    Args:
        checkpointer: Optional InMemorySaver instance for state persistence.
        enable_interrupt: If True, pauses before researchers for Human-in-the-Loop.
                         Set to False for automated web deployments (Streamlit).
    
    Returns:
        A compiled LangGraph StateGraph ready for invocation.
    """
    
    # Initialize the graph with our strict state schema
    workflow = StateGraph(ResearchState)
    
    # ========================================================================
    # 1. ADD NODES
    # ========================================================================
    
    workflow.add_node("planner", planner_node)
    workflow.add_node("synthesiser", synthesiser_node)
    workflow.add_node("critic", critic_node)
    
    workflow.add_node("researcher_1", create_researcher_node(0))
    workflow.add_node("researcher_2", create_researcher_node(1))
    workflow.add_node("researcher_3", create_researcher_node(2))
    
    # ========================================================================
    # 2. ADD EDGES (The Wiring)
    # ========================================================================
    
    workflow.add_edge(START, "planner")
    
    workflow.add_edge("planner", "researcher_1")
    workflow.add_edge("planner", "researcher_2")
    workflow.add_edge("planner", "researcher_3")
    
    workflow.add_edge("researcher_1", "synthesiser")
    workflow.add_edge("researcher_2", "synthesiser")
    workflow.add_edge("researcher_3", "synthesiser")
    
    workflow.add_edge("synthesiser", "critic")
    
    workflow.add_conditional_edges(
        "critic",
        should_continue,
        {
            "revise": "synthesiser",
            "end": END
        }
    )
    
    # ========================================================================
    # 3. COMPILE WITH PERSISTENCE AND CONDITIONAL INTERRUPT
    # ========================================================================
    
    if checkpointer is None:
        checkpointer = InMemorySaver()
        
    compile_args = {"checkpointer": checkpointer}
    
    if enable_interrupt:
        compile_args["interrupt_before"] = ["researcher_1"]
        
    return workflow.compile(**compile_args)

In [10]:
JUDGE_PROMPT = """You are an impartial research evaluator. Score the following research brief on a scale of 1-5 for each criterion.

ORIGINAL QUESTION: {question}

RESEARCH BRIEF:
{brief}

EVALUATE EACH CRITERION (1=Terrible, 5=Excellent):
1. Coverage: Does the brief address ALL parts of the question?
2. Faithfulness: Is every claim supported by a cited source (no hallucinations)?
3. Citation Quality: Are citations real, relevant, and properly formatted?
4. Coherence: Is it well-structured and readable?

Respond ONLY in JSON format:
{{
    "coverage": <int>,
    "faithfulness": <int>,
    "citation_quality": <int>,
    "coherence": <int>,
    "average_score": <float>
}}"""

def evaluate_brief(question: str, brief: str) -> dict:
    """Uses an LLM to grade a brief out of 5."""
    if not brief or len(brief) < 100:
        return {"coverage": 1, "faithfulness": 1, "citation_quality": 1, "coherence": 1, "average_score": 1.0}
        
    response = llm.invoke([HumanMessage(content=JUDGE_PROMPT.format(question=question, brief=brief))])
    
    try:
        # Extract JSON from the response
        content = response.content
        if "```" in content:
            content = content.split("```")[1]
            if content.startswith("json"):
                content = content[4:]
        return json.loads(content)
    except:
        return {"average_score": 3.0, "error": "Failed to parse evaluation"}

def run_evaluation():
    """Loads results, evaluates them, and prints the comparison table."""
    
    # Load data
    try:
        with open("./baseline_results.json", "r") as f:
            baseline_data = json.load(f)
    except FileNotFoundError:
        print("ERROR: baseline_results.json not found. Run baseline_agent.py first.")
        return

    try:
        with open("multiagent_metrics.json", "r") as f:
            multiagent_data = json.load(f)
    except FileNotFoundError:
        print("ERROR: multiagent_metrics.json not found. Run main.py first.")
        return

    results = []
    
    print("\n" + "=" * 90)
    print("EVALUATING BASELINE VS MULTI-AGENT SYSTEM")
    print("=" * 90)

    # Evaluate matching questions
    for b_res in baseline_data:
        question = b_res["question"]
        
        # Find matching multi-agent result
        m_res = next((m for m in multiagent_data if m["question"] == question), None)
        if not m_res:
            continue
            
        print(f"\nEvaluating: {question[:60]}...")
        
        b_eval = evaluate_brief(question, b_res["answer"])
        m_eval = evaluate_brief(question, m_res["draft"])
        
        row = {
            "topic": question[:45] + "...",
            "baseline_score": b_eval.get("average_score", 0),
            "multiagent_score": m_eval.get("average_score", 0),
            "baseline_time": b_res.get("time_seconds", 0),
            "multiagent_time": m_res.get("time_seconds", 0),
            "winner": "Multi-Agent" if m_eval.get("average_score", 0) > b_eval.get("average_score", 0) else "Baseline"
        }
        results.append(row)

    # Print Summary Table
    print("\n" + "-" * 90)
    print(f"{'TOPIC':<47} {'BASE':>6} {'MULTI':>6} {'TIME B':>7} {'TIME M':>7} {'WINNER':<12}")
    print("-" * 90)
    
    for r in results:
        print(f"{r['topic']:<47} {r['baseline_score']:>6.1f} {r['multiagent_score']:>6.1f} {r['baseline_time']:>6.0f}s {r['multiagent_time']:>6.0f}s {r['winner']:<12}")
        
    # Calculate Averages
    if results:
        avg_b_score = sum(r["baseline_score"] for r in results) / len(results)
        avg_m_score = sum(r["multiagent_score"] for r in results) / len(results)
        avg_b_time = sum(r["baseline_time"] for r in results) / len(results)
        avg_m_time = sum(r["multiagent_time"] for r in results) / len(results)
        
        print("-" * 90)
        print(f"{'AVERAGES':<47} {avg_b_score:>6.1f} {avg_m_score:>6.1f} {avg_b_time:>6.0f}s {avg_m_time:>6.0f}s")
        print(f"\nScore Improvement: +{avg_m_score - avg_b_score:.1f} points")
        print(f"Time Trade-off: +{avg_m_time - avg_b_time:.0f} seconds average")

    # Save evaluation results
    with open("evaluation_results.json", "w") as f:
        json.dump(results, f, indent=2)
        
    print("\nResults saved to evaluation_results.json")

In [ ]:
# Run the evaluation comparison
# Note: This requires baseline_results.json and multiagent_metrics.json to exist. 
# If you didn't run the full baseline earlier and multiagent_mterics, it will gracefully handle the missing file.
run_evaluation()


EVALUATING BASELINE VS MULTI-AGENT SYSTEM

Evaluating: Compare the current EV strategies of Tata Motors and BYD....

------------------------------------------------------------------------------------------
TOPIC                                             BASE  MULTI  TIME B  TIME M WINNER      
------------------------------------------------------------------------------------------
Compare the current EV strategies of Tata Mot...    2.8    4.8      7s      7s Multi-Agent 
------------------------------------------------------------------------------------------
AVERAGES                                           2.8    4.8      7s      7s

Score Improvement: +2.0 points
Time Trade-off: +-1 seconds average

Results saved to evaluation_results.json
